<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/sae/sae_features_d128.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformer-lens dictionary-learning


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 160.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 209.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 9.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 225.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 314.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 214.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 244.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 306.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 254.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 255.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 110.9 MB/s eta 0:0

In [ ]:
import torch
from transformer_lens import HookedTransformer, HookedTransformerConfig
import numpy as np
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader

from huggingface_hub import hf_hub_download


REPO_ID = "sebastianhoenig/2L2H_Final"
FILENAME = "D256_L2_H2_attnOnly1_lr5.0e-04_wd0.01.pt"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
try:
    from IPython.display import clear_output
    clear_output()
except ImportError:
    pass  # clear_output not available, continue anyway

import numpy as np
import pandas as pd
from tqdm import tqdm

E = 100  # num entities
T = 10   # num types/relations

SEP = E + T
Q = E + T + 1
PAD = E + T + 2
D_VOCAB = E + T + 3

IGNORE_INDEX = -100
ENTITIES = np.arange(0, E)
TYPES    = np.arange(E, E + T)

N_WORLDS = 80_000
MIN_FACTS, MAX_FACTS = 4, 8
SEED = 0

rng = np.random.default_rng(SEED)

def produce_example_by_index(idx: int, *, allow_self_loops: bool = False):
    rng = np.random.default_rng(np.random.SeedSequence([BASE_SEED, idx]))

    k = int(rng.integers(MIN_FACTS, MAX_FACTS + 1))

    facts = []
    seen_head_rel = set()
    while len(facts) < k:
        e = int(rng.integers(0, E))
        t = int(rng.integers(0, T)) + E
        if (e, t) in seen_head_rel:
            continue
        e2 = int(rng.integers(0, E))
        while (not allow_self_loops) and e2 == e:
            e2 = int(rng.integers(0, E))
        seen_head_rel.add((e, t))
        facts.append((e, t, e2))

    q_idx = int(rng.integers(0, k))
    Eq, Tq, E2q = facts[q_idx]

    if rng.random() < 0.75 and len(facts) < MAX_FACTS:
        distractor_t = int(rng.integers(0, T)) + E

        while distractor_t == Tq: # Ensure the relation is different
            distractor_t = int(rng.integers(0, T)) + E

        distractor_e2 = int(rng.integers(0, E))
        while distractor_e2 == E2q: # Ensure the tail is different
            distractor_e2 = int(rng.integers(0, E))

        # Add the distractor fact IF it doesn't create a collision
        if (Eq, distractor_t) not in seen_head_rel:
            distractor_fact = (Eq, distractor_t, distractor_e2)

            insert_pos = int(rng.integers(0, len(facts) + 1))
            facts.insert(insert_pos, distractor_fact)

    seq = []
    for (e, t, e2) in facts:
        seq.extend([e, t, e2, SEP])

    seq.extend([Tq, Eq, Q])

    label = E2q
    return seq, label
TOTAL_TRAIN = 16_100_000
BLOCK_SIZE  = 80_000
VAL_SIZE = 20_000
TRAIN_OFFSET = VAL_SIZE
TRAIN_SIZE   = TOTAL_TRAIN
BASE_SEED = 0

class ValDataset(torch.utils.data.Dataset):
    def __len__(self): return VAL_SIZE
    def __getitem__(self, i):
        seq, label = produce_example_by_index(i)
        return torch.tensor(seq, dtype=torch.long), torch.tensor(label, dtype=torch.long)


class TrainStream(torch.utils.data.IterableDataset):
    def __init__(self, block_size=BLOCK_SIZE, offset=TRAIN_OFFSET, size=TRAIN_SIZE):
        super().__init__()
        self.block_size = block_size
        self.offset = offset
        self.size = size
        self._epoch = 0

    def set_epoch(self, epoch:int):
        self._epoch = epoch

    def __iter__(self):
        # compute which block to serve this epoch, with wrap-around
        start_in_train = (self._epoch * self.block_size) % self.size
        # stream exactly block_size samples each epoch
        for i in range(self.block_size):
            local_idx = (start_in_train + i) % self.size
            global_idx = self.offset + local_idx
            seq, label = produce_example_by_index(global_idx)
            x = torch.tensor(seq, dtype=torch.long)
            y = torch.tensor(label, dtype=torch.long)
            yield x, y

    def __len__(self):
        return self.block_size

val_dataset = ValDataset()
train_dataset = TrainStream()

N_LAYERS = 2
HEADS = 2

d_model = 256
n_ctx   = 64

def build_model(n_layers: int, n_heads: int) -> HookedTransformer:
    if d_model % n_heads != 0:
        return None
    d_head = d_model // n_heads

    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_heads=n_heads,
        d_model=d_model,
        d_head=d_head,
        n_ctx=n_ctx,
        d_vocab=D_VOCAB,
        d_vocab_out=E,
        attn_only=True,
        normalization_type="LN",
        positional_embedding_type="rotary",
    )
    return HookedTransformer(cfg)

model = build_model(N_LAYERS, HEADS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
# Load the model
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
state_dict = pretrained_weights["model"]
model.load_state_dict(state_dict)

print("Model loaded successfully.")


def collate_fn(batch):
    max_len = max(len(seq) for seq,_ in batch)
    B = len(batch)
    toks   = torch.full((B, max_len), PAD, dtype=torch.long)
    target = torch.full((B, max_len), IGNORE_INDEX, dtype=torch.long)

    for i, (seq, label) in enumerate(batch):
        x = torch.tensor(seq if isinstance(seq, list) else seq.tolist(), dtype=torch.long)
        L = len(x)
        toks[i, :L] = x
        q_pos = (x == Q).nonzero(as_tuple=False).squeeze()
        assert q_pos.numel() == 1, "Each example must have exactly one Q"
        target[i, q_pos.item()] = int(label)
    return toks, target

In [ ]:
"""### Train a SAE using StandardTrainer with your datasets"""

import torch as t
from dictionary_learning.trainers.standard import StandardTrainer
from dictionary_learning import AutoEncoder, utils
from torch.utils.data import DataLoader

# === Pick activation site ===
layer = 1
act_name = f"blocks.{layer}.hook_resid_post"

# === SAE trainer ===
trainer = StandardTrainer(
    steps=100_000,                  # total training steps
    activation_dim=model.cfg.d_model,
    dict_size=128,                 # size of dictionary (SAE hidden dim) - reduced for faster training
    layer=layer,
    lm_name="entity_binding_model",  # just a string identifier
    lr=1e-4,
    l1_penalty=1e-3,
    device=device,
    warmup_steps=1000,             # learning rate warmup
    sparsity_warmup_steps=2000,    # sparsity penalty warmup
)

print(f"SAE initialized: dict_size={trainer.ae.dict_size}, activation_dim={trainer.ae.activation_dim}")

# === DataLoaders ===
train_loader = DataLoader(train_dataset, batch_size=64, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=64, collate_fn=collate_fn)

# === Training Loop ===
step = 0
best_loss = float('inf')
best_mse_loss = float('inf')
best_sparsity_loss = float('inf')
epochs_no_improve = 0
early_stopping_patience = 10

print("Starting SAE training...")

for epoch in range(200):   # loop over epochs
    epoch_loss = 0.0
    epoch_mse_loss = 0.0
    epoch_sparsity_loss = 0.0
    num_batches = 0

    # Set epoch for TrainStream
    train_dataset.set_epoch(epoch)

    for toks, _ in train_loader:
        toks = toks.to(device)

        # run model & collect activations
        with torch.no_grad():
            cache = model.run_with_cache(toks, names_filter=[act_name])[1]
            acts = cache[act_name]   # shape: [batch, seq, d_model]
            # Select activations for the last token in each sequence
            acts = acts[:, -1, :]  # shape: [batch, d_model]

        # update SAE
        trainer.update(step, acts)

        if step % 500 == 0:
            log = trainer.loss(acts, step, logging=True)
            current_loss = log.losses['loss']
            current_mse = log.losses['mse_loss']
            current_sparsity = log.losses['sparsity_loss']
            print(f"Step {step}: loss={current_loss:.4f}, mse={current_mse:.4f}, sparsity={current_sparsity:.4f}")

            # Accumulate loss for epoch tracking
            epoch_loss += current_loss
            epoch_mse_loss += current_mse
            epoch_sparsity_loss += current_sparsity
            num_batches += 1

        step += 1
        if step >= trainer.steps:
            break

    if num_batches > 0:
        avg_epoch_loss = epoch_loss / num_batches
        avg_epoch_mse_loss = epoch_mse_loss / num_batches
        avg_epoch_sparsity_loss = epoch_sparsity_loss / num_batches

        print(f"Epoch {epoch}: Avg Loss={avg_epoch_loss:.4f}, Avg MSE={avg_epoch_mse_loss:.4f}, Avg Sparsity={avg_epoch_sparsity_loss:.4f}")

        # Check for improvement
        if avg_epoch_loss < best_loss or avg_epoch_mse_loss < best_mse_loss or avg_epoch_sparsity_loss < best_sparsity_loss:
            best_loss = min(best_loss, avg_epoch_loss)
            best_mse_loss = min(best_mse_loss, avg_epoch_mse_loss)
            best_sparsity_loss = min(best_sparsity_loss, avg_epoch_sparsity_loss)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epochs.")

        if epochs_no_improve >= early_stopping_patience:
            print(f"Early stopping triggered after {epoch + 1} epochs.")
            break

    if step >= trainer.steps:
        break


print("SAE training completed!")

SAE initialized: dict_size=128, activation_dim=256
Starting SAE training...
Step 0: loss=72.0052, mse=72.0052, sparsity=3.2529
Step 500: loss=3.8952, mse=3.8814, sparsity=55.0258
Step 1000: loss=2.1115, mse=2.0821, sparsity=58.8713
Epoch 0: Avg Loss=26.0040, Avg MSE=25.9896, Avg Sparsity=39.0500
Step 1500: loss=0.3774, mse=0.3301, sparsity=63.1603
Step 2000: loss=0.3611, mse=0.2984, sparsity=62.6746
Epoch 1: Avg Loss=0.3693, Avg MSE=0.3143, Avg Sparsity=62.9175
Step 2500: loss=0.3342, mse=0.2739, sparsity=60.2895
Step 3000: loss=0.2447, mse=0.1830, sparsity=61.6947
Step 3500: loss=0.2425, mse=0.1854, sparsity=57.1399
Epoch 2: Avg Loss=0.2738, Avg MSE=0.2141, Avg Sparsity=59.7080
Step 4000: loss=0.1788, mse=0.1195, sparsity=59.3227
Step 4500: loss=0.1690, mse=0.1130, sparsity=55.9806
Epoch 3: Avg Loss=0.1739, Avg MSE=0.1162, Avg Sparsity=57.6516
Step 5000: loss=0.1393, mse=0.0867, sparsity=52.5889
Step 5500: loss=0.1004, mse=0.0497, sparsity=50.7124
Step 6000: loss=0.0822, mse=0.0327, s

In [ ]:
# === Save trained SAE ===
save_path = f"{act_name}_trained.pt"
t.save(trainer.ae.state_dict(), save_path)
print(f"SAE saved to {save_path}")

SAE saved to blocks.1.hook_resid_post_trained.pt


In [ ]:
# === Evaluate on validation set ===
print("\nEvaluating SAE on validation set...")
trainer.ae.eval()
val_losses = []
val_sparsity = []

with torch.no_grad():
    for toks, _ in val_loader:
        toks = toks.to(device)
        cache = model.run_with_cache(toks, names_filter=[act_name])[1]
        acts = cache[act_name]
        # acts = acts.reshape(-1, acts.size(-1))

        # Select activations for the last token in each sequence
        acts = acts[:, -1, :]  # shape: [batch, d_model]

        # Get features and reconstruction
        features = trainer.ae.encode(acts)
        reconstruction = trainer.ae.decode(features)

        # Compute metrics
        mse_loss = torch.nn.functional.mse_loss(acts, reconstruction)
        sparsity = (features == 0).float().mean()

        val_losses.append(mse_loss.item())
        val_sparsity.append(sparsity.item())

avg_val_loss = np.mean(val_losses)
avg_val_sparsity = np.mean(val_sparsity)
print(f"Validation MSE: {avg_val_loss:.6f}")
print(f"Validation Sparsity: {avg_val_sparsity:.4f} ({avg_val_sparsity*100:.2f}%)")


Evaluating SAE on validation set...
Validation MSE: 0.000009
Validation Sparsity: 0.1298 (12.98%)


In [ ]:
acts.shape

torch.Size([32, 256])

In [ ]:
len(val_dataset)

20000

In [ ]:
all_labels = []
for i in range(len(val_dataset)):
    all_labels.append(val_dataset[i][1])
all_labels = torch.stack(all_labels)


In [ ]:
all_labels.shape

torch.Size([20000])

In [ ]:
# === Extract SAE features from validation set ===
print("\nExtracting SAE features from validation set...")
all_features = []
all_activations = []
all_reconstructions = []
with torch.no_grad():
    for toks, _ in val_loader:
        toks = toks.to(device)
        cache = model.run_with_cache(toks, names_filter=[act_name])[1]
        acts = cache[act_name]
        # acts = acts.reshape(-1, acts.size(-1))

        # Select activations for the last token in each sequence
        acts = acts[:, -1, :]  # shape: [batch, d_model]
        # Extract features
        features = trainer.ae.encode(acts)
        reconstruction = trainer.ae.decode(features)

        all_features.append(features.cpu())
        all_activations.append(acts.cpu())
        all_reconstructions.append(reconstruction.cpu())

# Concatenate all features
all_features = torch.cat(all_features, dim=0)
all_activations = torch.cat(all_activations, dim=0)
all_reconstructions = torch.cat(all_reconstructions, dim=0)

print(f"Extracted features shape: {all_features.shape}")
print(f"Original activations shape: {all_activations.shape}")
print(f"Reconstructions shape: {all_reconstructions.shape}")
print(f"Labels shape: {all_labels.shape}")


Extracting SAE features from validation set...
Extracted features shape: torch.Size([20000, 128])
Original activations shape: torch.Size([20000, 256])
Reconstructions shape: torch.Size([20000, 256])
Labels shape: torch.Size([20000])


In [ ]:
val_dataset[0][0].shape

torch.Size([35])

In [ ]:
# === Analyze feature statistics ===
print("\n=== SAE Feature Analysis ===")
sparsity = (all_features == 0).float().mean()
print(f"Overall feature sparsity: {sparsity:.4f} ({sparsity*100:.2f}% of features are zero)")

# Feature activation frequency
feature_activations = (all_features > 0).float()
feature_frequency = feature_activations.mean(dim=0)
print(f"\nTop 10 most active features:")
top_features = feature_frequency.topk(10)
for i, (idx, freq) in enumerate(zip(top_features.indices, top_features.values)):
    print(f"  Feature {idx.item()}: {freq.item():.4f}")

# Sparsity per sample
sparsity_per_sample = (all_features == 0).float().mean(dim=1)
print(f"\nSparsity per sample - Mean: {sparsity_per_sample.mean():.4f}, Std: {sparsity_per_sample.std():.4f}")

# Reconstruction quality
reconstruction_error = torch.norm(all_activations - all_reconstructions, dim=1)
print(f"Reconstruction error - Mean: {reconstruction_error.mean():.4f}, Std: {reconstruction_error.std():.4f}")

print(f"\n✅ SAE training and feature extraction completed!")
print(f"Features saved in 'all_features' tensor with shape {all_features.shape}")
print(f"You can now use these features for interpretability analysis or downstream tasks.")

# Save features to file
features_save_path = f"{act_name}_features.pt"
torch.save({
    'features': all_features,
    'activations': all_activations,
    'labels': all_labels,
    'reconstructions': all_reconstructions,
    'feature_frequency': feature_frequency,
    'sparsity_per_sample': sparsity_per_sample,
    'reconstruction_error': reconstruction_error
}, features_save_path)
print(f"Features saved to {features_save_path}")


=== SAE Feature Analysis ===
Overall feature sparsity: 0.1298 (12.98% of features are zero)

Top 10 most active features:
  Feature 89: 0.9895
  Feature 29: 0.9883
  Feature 94: 0.9878
  Feature 32: 0.9804
  Feature 95: 0.9568
  Feature 81: 0.9501
  Feature 90: 0.9489
  Feature 63: 0.9451
  Feature 71: 0.9443
  Feature 4: 0.9439

Sparsity per sample - Mean: 0.1298, Std: 0.0508
Reconstruction error - Mean: 0.0405, Std: 0.0266

✅ SAE training and feature extraction completed!
Features saved in 'all_features' tensor with shape torch.Size([20000, 128])
You can now use these features for interpretability analysis or downstream tasks.
Features saved to blocks.1.hook_resid_post_features.pt


In [ ]:
from collections import defaultdict, Counter

# === Analyze correlation between features and labels ===
print("\n=== Feature-Label Correlation Analysis ===")

# Find the index of the most activated feature for each sample
most_active_feature_indices = torch.argmax(all_features, dim=1)

# Check the shapes to ensure they match
print(f"Shape of most_active_feature_indices: {most_active_feature_indices.shape}")
print(f"Shape of all_labels: {all_labels.shape}")

labels_to_sae_idx = defaultdict(list)
for feat_idx, label in zip(most_active_feature_indices, all_labels):
  labels_to_sae_idx[label.item()].append(feat_idx.item())
labels_to_sae_idx_cnt = defaultdict(Counter)
for i, occurrences in labels_to_sae_idx.items():
    labels_to_sae_idx_cnt[i] = Counter(occurrences).most_common() # feat_idx, counts


=== Feature-Label Correlation Analysis ===
Shape of most_active_feature_indices: torch.Size([20000])
Shape of all_labels: torch.Size([20000])


In [ ]:
labels_to_sae_idx_cnt

defaultdict(collections.Counter,
            {26: [(32, 135), (124, 59)],
             25: [(32, 139), (124, 72)],
             38: [(32, 139), (124, 67), (29, 1)],
             86: [(32, 126), (124, 64)],
             65: [(32, 131), (124, 65)],
             98: [(32, 140), (124, 77)],
             21: [(32, 125), (124, 75)],
             53: [(32, 131), (124, 67)],
             5: [(32, 129), (124, 62), (29, 1)],
             46: [(32, 146), (124, 71)],
             80: [(32, 130), (124, 63), (29, 1)],
             96: [(32, 146), (124, 68)],
             75: [(32, 120), (124, 72)],
             58: [(32, 152), (124, 68)],
             24: [(32, 117), (83, 79)],
             74: [(32, 128), (117, 71)],
             57: [(32, 132), (26, 72)],
             64: [(32, 136), (124, 71)],
             8: [(32, 108), (124, 79), (29, 1)],
             12: [(32, 118), (124, 60)],
             31: [(32, 110), (124, 87)],
             19: [(32, 136), (124, 63)],
             91: [(32, 146), (26,